In [1]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import optuna 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader 
from sklearn.model_selection import train_test_split
from DataEncoder import encode_pad_event, encode_pad_sequence, scale_time_differences, encode_label_event, node_time_list, scale_time_differences_fast_fixed, length_stratified_split
from GATConv import prepare_data_core_timedif, prepare_data_y, CustomDataset, EarlyStopping, train, evaluate, custom_collate_fn, DualGATModel
from GATConv import predict, top_k_accuracy, predict_per_sequence, average_bleu_score, compute_dls_and_exact_match, sequence_level_top_k_accuracy, analyze_sequence_errors, predict_per_sequence_with_probs, sequence_level_top_k_analysis, show_error_sequences
import os 
import shutil

In [2]:
#event = pd.read_csv("../output/BPI12.csv")
event = pd.read_csv("../output/BPI12w.csv")
#event = pd.read_csv("../output/BPI13i.csv")
#event = pd.read_csv("../output/BPI13c.csv")
#event = pd.read_csv("../output/helpdesk.csv")

In [3]:
# check the size of sequence
shortest_sequence = event.groupby('sequence').size().min()
longest_sequence = event.groupby('sequence').size().max()
print('shortest_sequence:', shortest_sequence)
print('longest_sequence:', longest_sequence)
event = event[event.groupby('sequence')['sequence'].transform('size') >= 2].reset_index(drop=True)

shortest_sequence: 2
longest_sequence: 156


In [4]:
#core_event = 'event_label' 
core_event = 'event' # with status
case_index = 'sequence'

In [5]:
core_encode, y_encode, core_size, output_size, le_event = encode_label_event(event, core_event, case_index)

In [6]:
#BPI12 
#event['ec1'] = event['ec1'].astype(str) 
cat_col_event = ['ec1']
#BPI17
#cat_col_event = ['ec1', 'ec2', 'ec3']
#BPI13
#cat_col_event = ['ec1', 'ec2', 'ec4', 'ec5']
#helpdesk
#cat_col_event = ['ec1']
num_col_event = []
event_encode = encode_pad_event(event, cat_col_event, num_col_event, case_index, cat_mask = True, num_mask = True, eos = False)

In [7]:
#BPI12
sequence = event[['sequence','sn1']].groupby('sequence').first()
#BPI17
#sequence = event[['sequence','sn1', 'sc1', 'sc2']].groupby('sequence').first()
#BPI13
#sequence = event[['sequence','sc3', 'sc1', 'sc2']].groupby('sequence').first()
#sequence = event[['sequence','sc1']].groupby('sequence').first()
sequence = sequence.reset_index()
#sequence = event.groupby('sequence').apply(lambda x: x.iloc[prefix_size - 1:]).reset_index(drop=True)

In [8]:
#bpi12
cat_col_seq = []
num_col_seq = ['sn1']
#Bpi13
#cat_col_seq = ['sc1','sc2','sc3']
#num_col_seq = []
#helpdesk
#cat_col_seq = ['sc1']
#num_col_seq = []
sequence_encode = encode_pad_sequence(sequence, cat_col_seq, num_col_seq)

In [9]:
start_time_col = 'time'
#start_time_col = 'Start Timestamp'
scaled_time_diffs = scale_time_differences_fast_fixed(event, sequence, start_time_col, case_index)

In [10]:
max_num_events = event_encode.shape[1]
# Expand sequence features to match the shape of event features
sequence_features_expanded = np.expand_dims(sequence_encode, axis=1)
sequence_features_expanded = np.repeat(sequence_features_expanded, max_num_events, axis=1)

# Combine event and sequence features
combined_features = np.concatenate((event_encode, sequence_features_expanded), axis=2)

In [11]:
num_sequences = event_encode.shape[0]
num_event_features = combined_features.shape[2]
num_embedding_features = core_size

In [12]:
node_times = node_time_list(event, start_time_col, case_index)

In [13]:
# graph data preparation
event_feature_list = prepare_data_core_timedif(combined_features, core_encode, scaled_time_diffs, node_times)
y_list = prepare_data_y(combined_features, y_encode)

In [14]:
# Use stratified sampling over sequence length to preserve distributional characteristics
train_indices, test_indices = length_stratified_split(event_feature_list, test_size=0.2, n_bins=10)

# Split the data
train_event_features = [event_feature_list[i] for i in train_indices]
test_event_features = [event_feature_list[i] for i in test_indices]
train_y = [y_list[i] for i in train_indices]
test_y = [y_list[i] for i in test_indices]

# Print statistics
train_lengths = [event_feature_list[i].x.shape[0] for i in train_indices]
test_lengths = [event_feature_list[i].x.shape[0] for i in test_indices]

print(f"Train set: {len(train_indices)} samples")
print(f"Train length range: {min(train_lengths)} - {max(train_lengths)}")
print(f"Train length mean: {np.mean(train_lengths):.2f}")

print(f"\nTest set: {len(test_indices)} samples") 
print(f"Test length range: {min(test_lengths)} - {max(test_lengths)}")
print(f"Test length mean: {np.mean(test_lengths):.2f}")

Train set: 7730 samples
Train length range: 2 - 154
Train length mean: 15.87

Test set: 1928 samples
Test length range: 13 - 156
Test length mean: 24.59


In [15]:
train_dataset = CustomDataset(train_event_features, train_y)
test_dataset = CustomDataset(test_event_features, test_y)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# parameters
embedding_dims = 64
gat_hidden_dim_event = 32
gat_hidden_dim_embed = 128
gat_hidden_dim_concat = 256
output_dim = output_size  # vocab size
num_heads = 4

model = DualGATModel(
    num_event_features=num_event_features,
    num_embedding_features=num_embedding_features,
    embedding_dims=embedding_dims,
    gat_hidden_dim_event=gat_hidden_dim_event,
    gat_hidden_dim_embed=gat_hidden_dim_embed,
    gat_hidden_dim_concat=gat_hidden_dim_concat,
    output_dim=output_dim,
    num_heads=num_heads
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-1)  # ignore padding

In [17]:
batch_size = 16  
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

In [19]:
early_stopping = EarlyStopping(patience=3, delta=0.0)
num_epochs = 10

# Filepath to save model
model_path = "../output/models/BPI12_timeedge_es.pt"

config = {
    'num_event_features': num_event_features,
    'num_embedding_features': num_embedding_features,
    'embedding_dims': embedding_dims,
    'gat_hidden_dim_event': gat_hidden_dim_event,
    'gat_hidden_dim_embed': gat_hidden_dim_embed,
    'gat_hidden_dim_concat': gat_hidden_dim_concat,
    'output_dim': output_dim,
    'num_heads': num_heads
}

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)

Epoch 1/10 | Train Loss: 0.4784 Acc: 0.8026 | Test Loss: 0.3906 Acc: 0.8646
New best model at epoch 1, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 2/10 | Train Loss: 0.3717 Acc: 0.8614 | Test Loss: 0.4122 Acc: 0.8318
Epoch 3/10 | Train Loss: 0.3703 Acc: 0.8615 | Test Loss: 0.3727 Acc: 0.8665
New best model at epoch 3, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 4/10 | Train Loss: 0.3633 Acc: 0.8642 | Test Loss: 0.3395 Acc: 0.8737
New best model at epoch 4, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 5/10 | Train Loss: 0.3488 Acc: 0.8699 | Test Loss: 0.3377 Acc: 0.8707
New best model at epoch 5, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 6/10 | Train Loss: 0.3510 Acc: 0.8702 | Test Loss: 0.3303 Acc: 0.8731
New best model at epoch 6, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 7/10 | Train Loss: 0.3393 Acc: 0.8718 | Test Loss: 0.3350 Acc: 0.8749
Epoch 8/10 | Train Loss: 0.3559 Acc: 0.8699 | Test Loss: 0.5449 Acc: 0.8481
Epoch 9/10 | Trai

In [20]:
preds, labels, outs = predict(model, test_loader, device)

# Check some predictions
for i in range(10):
    print(f"Predicted: {preds[i].item()} | Actual: {labels[i].item()}")

Predicted: 9 | Actual: 9
Predicted: 7 | Actual: 14
Predicted: 7 | Actual: 7
Predicted: 15 | Actual: 15
Predicted: 13 | Actual: 13
Predicted: 15 | Actual: 15
Predicted: 13 | Actual: 13
Predicted: 15 | Actual: 15
Predicted: 17 | Actual: 17
Predicted: 13 | Actual: 13


In [21]:
top1 = top_k_accuracy(outs, labels, k=1)
top3 = top_k_accuracy(outs, labels, k=3)
top5 = top_k_accuracy(outs, labels, k=5)

print(f"Top-1 Acc: {top1:.4f} | Top-3 Acc: {top3:.4f} | Top-5 Acc: {top5:.4f}")

Top-1 Acc: 0.8777 | Top-3 Acc: 0.9996 | Top-5 Acc: 0.9998


In [22]:
preds_seq, labels_seq = predict_per_sequence(model, test_loader, device)

In [23]:
bleu = average_bleu_score(preds_seq, labels_seq)
print(f"Average BLEU Score: {bleu:.4f}")

Average BLEU Score: 0.7239


In [24]:
avg_dls, exact_acc = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"Damerau-Levenshtein Similarity (avg): {avg_dls:.4f}")
print(f"Exact Match Accuracy: {exact_acc:.4f}")


Damerau-Levenshtein Similarity (avg): 0.8667
Exact Match Accuracy: 0.0000


In [25]:
# Usage
seq_top3_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=3)
print(f"Sequence-Level Top-3 Accuracy: {seq_top3_acc:.4f}")

seq_top5_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=5)
print(f"Sequence-Level Top-5 Accuracy: {seq_top5_acc:.4f}")

Sequence-Level Top-3 Accuracy: 0.9943
Sequence-Level Top-5 Accuracy: 0.9969


In [26]:
# Usage example
pos_errors, error_types, seq_stats = analyze_sequence_errors(model, test_loader, device, k=3)


=== Error Analysis Report ===
Total sequences analyzed: 1928
Total sequences with errors: 1928
Sequence length stats: {'min': 13, 'max': 156, 'mean': np.float64(24.58713692946058), 'median': np.float64(17.0)}

Error distribution by position:
Position 0: 0 errors (0.0% of all errors)
Position 1: 925 errors (16.0% of all errors)
Position 2: 120 errors (2.1% of all errors)
Position 3: 370 errors (6.4% of all errors)
Position 4: 293 errors (5.1% of all errors)
Position 5: 181 errors (3.1% of all errors)
Position 6: 212 errors (3.7% of all errors)
Position 7: 140 errors (2.4% of all errors)
Position 8: 146 errors (2.5% of all errors)
Position 9: 132 errors (2.3% of all errors)
Position 10: 140 errors (2.4% of all errors)
Position 11: 132 errors (2.3% of all errors)
Position 12: 285 errors (4.9% of all errors)
Position 13: 275 errors (4.7% of all errors)
Position 14: 255 errors (4.4% of all errors)
Position 15: 278 errors (4.8% of all errors)
Position 16: 198 errors (3.4% of all errors)
Pos

In [27]:
# First get predictions with probabilities
all_preds, all_labels, all_topk = predict_per_sequence_with_probs(model, test_loader, device, k=3)

# Then analyze sequence-level top-k accuracy
topk_acc, error_stats = sequence_level_top_k_analysis(all_topk, all_labels)

print(f"Sequence-Level Top-3 Accuracy: {topk_acc:.4f}")
print("\nError Analysis:")
print(f"- Most error-prone positions: {error_stats['position_errors']}")
print(f"- Top prediction mistakes: {error_stats['top_errors']}")

Sequence-Level Top-3 Accuracy: 0.9943

Error Analysis:
- Most error-prone positions: {0: np.int64(0), 1: np.int64(0), 2: np.int64(0), 3: np.int64(0), 4: np.int64(1), 5: np.int64(1), 6: np.int64(0), 7: np.int64(0), 8: np.int64(0), 9: np.int64(0), 10: np.int64(3), 11: np.int64(2), 12: np.int64(2), 13: np.int64(0), 14: np.int64(0), 15: np.int64(1), 16: np.int64(1), 17: np.int64(0), 18: np.int64(1), 19: np.int64(1), 20: np.int64(0), 21: np.int64(1), 22: np.int64(0), 23: np.int64(0), 24: np.int64(0), 25: np.int64(1), 26: np.int64(0), 27: np.int64(0), 28: np.int64(1), 29: np.int64(0), 30: np.int64(0), 31: np.int64(2), 32: np.int64(0), 33: np.int64(0), 34: np.int64(0), 35: np.int64(0), 36: np.int64(0), 37: np.int64(1), 38: np.int64(1)}
- Top prediction mistakes: {(18, 11): 3, (0, 6): 2, (12, 10): 2, (0, 19): 1, (0, 5): 1}


In [28]:
show_error_sequences(all_topk, all_labels, num=3)


Error Sequence #90:
True: [9, 14, 7, 15, 13, 15, 17, 13, 18, 16, 19, 19, 0]
Pred: [9, 7, 7, 15, 13, 15, 17, 13, 18, 16, 0, 19, 19]
Mismatches:
Pos 10: True 19 not in [0, 18, 9]

Error Sequence #898:
True: [9, 14, 7, 15, 13, 15, 17, 13, 18, 16, 5, 6, 17, 4, 18, 16, 0]
Pred: [9, 7, 7, 15, 13, 15, 13, 13, 18, 16, 0, 0, 6, 4, 18, 11, 0]
Mismatches:
Pos 10: True 5 not in [0, 18, 9]
Pos 11: True 6 not in [0, 16, 18]
Pos 12: True 17 not in [6, 4, 1]

Error Sequence #939:
True: [9, 14, 7, 15, 13, 15, 14, 13, 15, 17, 13, 18, 16, 18, 16, 16, 0]
Pred: [9, 7, 7, 15, 13, 15, 13, 13, 15, 17, 13, 18, 16, 0, 16, 0, 19]
Mismatches:
Pos 15: True 16 not in [0, 18, 19]


In [29]:
from sklearn.metrics import classification_report
unique_true = np.unique(labels.cpu().numpy())
unique_preds = np.unique(preds.cpu().numpy())

print("Classes in true labels:", unique_true)
print("Classes in predictions:", unique_preds)
print("Missing classes:", set(unique_true) - set(unique_preds))

actual_classes = np.union1d(unique_true, unique_preds)
report = classification_report(
    labels.cpu().numpy(),
    preds.cpu().numpy(),
    target_names=[f"Class_{i}" for i in actual_classes] ,
    digits=4
)
print(report)

Classes in true labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
Classes in predictions: [ 0  1  3  4  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
Missing classes: {np.int64(2), np.int64(5)}
              precision    recall  f1-score   support

     Class_0     0.6032    0.4487    0.5146      1928
     Class_1     0.6187    0.9507    0.7495       872
     Class_2     0.0000    0.0000    0.0000        16
     Class_3     0.9970    0.7569    0.8605       872
     Class_4     0.7683    0.9844    0.8630        64
     Class_5     0.0000    0.0000    0.0000        18
     Class_6     0.9388    0.7188    0.8142        64
     Class_7     0.8084    0.9818    0.8867      6698
     Class_8     0.7381    0.1978    0.3119       627
     Class_9     0.9499    0.9998    0.9742      6582
    Class_10     0.9247    0.9593    0.9417      3147
    Class_11     0.3774    0.1205    0.1826       664
    Class_12     0.9671    0.9908    0.9788      3146
    Class_13     0.9183    0.9669

c:\Program Files\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Program Files\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Program Files\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [30]:
# Load model state
checkpoint = torch.load(model_path, map_location=device)
config = checkpoint['config']

# Rebuild the model using the same config
model = DualGATModel(
    num_event_features=config['num_event_features'],
    num_embedding_features=config['num_embedding_features'],
    embedding_dims=config['embedding_dims'],
    gat_hidden_dim_event=config['gat_hidden_dim_event'],
    gat_hidden_dim_embed=config['gat_hidden_dim_embed'],
    gat_hidden_dim_concat=config['gat_hidden_dim_concat'],
    output_dim=config['output_dim'],
    num_heads=config['num_heads']
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

# Rebuild optimizer (if needed)
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Recover attention and event IDs
start_epoch = checkpoint['epoch'] + 1 

for epoch in range(start_epoch, 10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)